# Задание 5. 


Написать SQL-запросы на основе описанных таблиц в Clickhouse
Рассчитать WAU скользящим окном в неделю с шагом в 1 день, при этом текущая дата должна включаться в расчет.
WAU - кол-во активных пользователей сайта.
Например, для даты 19.12.2023 нужно рассчитать WAU за период 13.12.2023-19.12.2023,
для  20.12.2023  - 14.12.2023-20.12.2023
для  21.12.2023  - 15.12.2023-21.12.2023
и т.д.

**Допущение**: начинаем считать после накопления семи дней 

In [ ]:
import clickhouse_connect
import pandas as pd

# для выполнения запросов укажите параметры подключения к ClickHouse
client = clickhouse_connect.get_client(host=, port=, username=, password=)
print("Успешное подключение!")

In [ ]:
sql_query = '''
WITH 
dates AS (
    SELECT 
        toDate(min(timestamp)) AS min_d, 
        toDate(max(timestamp)) AS max_d 
    FROM dwh.site_activity
),
calendar AS (
    SELECT min_d + number AS date
    FROM dates
    ARRAY JOIN range(toUInt32(max_d - min_d + 1)) AS number
),
calendar_windows AS (
    SELECT 
        date,
        arrayJoin(arrayMap(x -> date - x, range(7))) AS join_date
    FROM calendar
),
active_users_by_day AS (
    SELECT
        toDate(timestamp) AS activity_date,
        user_id
    FROM dwh.site_activity
    WHERE user_id IS NOT NULL
    GROUP BY
        activity_date,
        user_id
),
wau_calc AS (
    SELECT 
        w.date,
        uniqExact(a.user_id) AS WAU
    FROM calendar_windows AS w
    INNER JOIN active_users_by_day AS a
        ON w.join_date = a.activity_date
    GROUP BY w.date
)
SELECT 
    c.date,
    coalesce(w.WAU, 0) AS WAU
FROM calendar AS c
LEFT JOIN wau_calc AS w 
    ON c.date = w.date
WHERE c.date >= (
    SELECT toDate(min(timestamp)) + 6
    FROM dwh.site_activity
)
ORDER BY c.date
'''

df = client.query_df(sql_query)
display(df)